# step-counter-increment composite — cx17: step counter increments after zero_grad(set_to_none=True)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `step-counter-increment`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "step-counter-increment"
DD_ATOM_IDS = ["step-counter-increment", "zero-grad-set-none"]
DD_SUBTOPICS = ["Trainer: step counter increment", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The canonical training-step ritual ends with `zero_grad`, then ticks the counter:

```python
loss.backward()
optimizer.step()
optimizer.zero_grad(set_to_none=True)   # atom A.
self.step += 1                          # atom B (AFTER zero_grad).
```

**Atom A — `zero-grad-set-none`.** Sets every `p.grad = None`. Same atom as cx13. The next `.backward()` allocates fresh grads — no accumulation.

**Atom B — `step-counter-increment`.** The counter advances once per committed update. Same atom as cx14/cx16. Position it AFTER `zero_grad` so that, at the moment the counter shows `N`, the model has been updated N times AND its gradient slate is clean for batch N+1.

**Why this exact order.** If you increment BEFORE `zero_grad`, then any code that reads `self.step` between the increment and the `zero_grad` sees a state where 'we've done N updates' but `p.grad` still holds batch N's gradient. That gradient is about to be wiped — so it's stale information masquerading as fresh — and any concurrent reader (logging thread, gradient-clipping monitor, gradient-norm logger) sees inconsistent state. The rule: tick LAST.

We test by asserting BOTH semantic facts after each step: counter incremented by exactly 1 AND every `p.grad is None`.

### Composite Exercise — step counter increments after zero_grad(set_to_none=True)

**Atoms exercised together**: `step-counter-increment`, `zero-grad-set-none`

Implement `cx17_train_loop(model, optimizer, loader, loss_fn, start_step)`. ONE epoch.

For each `(x, y)`:
1. Forward + backward + optimizer.step (same as cx13/cx14).
2. `optimizer.zero_grad(set_to_none=True)` (atom A — explicit `set_to_none=True`).
3. `step += 1` (atom B — AFTER `zero_grad`).

Return `(final_step, snapshots)` where `snapshots` is a list of `(step_at_end_of_batch, all_grads_are_none)` tuples — one per batch. The boolean is `all(p.grad is None for p in model.parameters())`.

**Test asserts**:
- Step sequence is `[start_step+1, start_step+2, ...]`.
- After EVERY batch (i.e. at every snapshot), all grads are None.
- After the final batch returns, every `p.grad is None` (set_to_none observed externally).
- Model parameters MOVED (so step did run before zero_grad).

In [ ]:
def cx17_train_loop(model, optimizer, loader, loss_fn, start_step):
    step = start_step
    snaps = []
    for x, y in loader:
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        # Atom A: zero-grad-set-none. Wipe grads.
        optimizer.zero_grad(set_to_none=True)
        # Atom B: counter tick AFTER zero_grad.
        step += 1
        all_none = all(p.grad is None for p in model.parameters())
        snaps.append((step, all_none))
    return step, snaps


<details><summary>Show solution — cx17</summary>

```python
def cx17_train_loop(model, optimizer, loader, loss_fn, start_step):
    step = start_step
    snaps = []
    for x, y in loader:
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        # Atom A: zero-grad-set-none. Wipe grads.
        optimizer.zero_grad(set_to_none=True)
        # Atom B: counter tick AFTER zero_grad.
        step += 1
        all_none = all(p.grad is None for p in model.parameters())
        snaps.append((step, all_none))
    return step, snaps
```

The snapshot is taken AFTER both atoms have fired, so `all_none == True` is the steady state contract. If you reorder to `increment → zero_grad`, the snapshot would still pass — but Case E (no accumulation) catches the deeper bug of forgetting `zero_grad` entirely. The canonical order `step → zero_grad → counter` is what makes the step counter a trustworthy 'how many committed updates so far' invariant.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["Trainer: step counter increment", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()